# Amazon Sales -- SQL Analysis

Loads `../data/amazon_sales_data.csv` into a real SQLite database (schema in `schema.sql`) and
runs the business queries in `queries.sql` against it, so this is genuine SQL execution -- not a
Python re-implementation of what SQL would do. Complements `../notebooks/python_eda_analysis.ipynb`
(pandas-based EDA) and the Power BI dashboard: same dataset, three different tools, matching what
this repo's name promises.

In [1]:
import sqlite3
import re
import pandas as pd

DB_PATH = ":memory:"
conn = sqlite3.connect(DB_PATH)

## 1. Load the CSV, reshape dates to ISO-8601, load into SQLite per `schema.sql`

In [2]:
df = pd.read_csv("../data/amazon_sales_data.csv")

# The source CSV stores dates as M/D/YYYY, but 34 rows use "M-D-YYYY" (hyphens instead of
# slashes) -- a real inconsistency in the raw data, not a day-first/month-first ambiguity: every
# one of those 34 rows has a first component <=12, so they're the same M-D-YYYY format, just a
# different separator, not genuinely day-first dates. Normalize the separator, then parse
# uniformly. SQLite's date functions (JULIANDAY, STRFTIME) expect ISO-8601 (YYYY-MM-DD), so
# convert on the way in rather than inside SQL.
def parse_mixed_date(series):
    normalized = series.str.replace("-", "/", regex=False)
    return pd.to_datetime(normalized, format="%m/%d/%Y").dt.strftime("%Y-%m-%d")

df["Order Date"] = parse_mixed_date(df["Order Date"])
df["Ship Date"] = parse_mixed_date(df["Ship Date"])

df = df.rename(columns={
    "Region": "region", "Country": "country", "Item Type": "item_type",
    "Sales Channel": "sales_channel", "Order Priority": "order_priority",
    "Order Date": "order_date", "Order ID": "order_id", "Ship Date": "ship_date",
    "Units Sold": "units_sold", "Unit Price": "unit_price", "Unit Cost": "unit_cost",
    "Total Revenue": "total_revenue", "Total Cost": "total_cost", "Total Profit": "total_profit",
})

with open("schema.sql") as f:
    conn.executescript(f.read())

df.to_sql("sales", conn, if_exists="append", index=False)

pd.read_sql("SELECT COUNT(*) AS row_count FROM sales", conn)

,row_count
0,100


## 2. Run each query from `queries.sql` against the real database

In [3]:
def load_queries(path):
    with open(path) as f:
        text = f.read()
    # split into blocks starting at each "-- N. <title>" comment
    blocks = re.split(r"\n(?=-- \d+\.\s)", text)
    queries = []
    for block in blocks:
        block = block.strip()
        if not re.match(r"-- \d+\.\s", block):
            continue  # the file's header comment before query 1, not a real query
        title_line = block.splitlines()[0]
        title = title_line.lstrip("- ").strip()
        sql = "\n".join(l for l in block.splitlines() if not l.strip().startswith("--")).strip()
        assert sql, f"query {title!r} parsed to empty SQL -- parser bug"
        queries.append((title, sql))
    return queries

queries = load_queries("queries.sql")
print(f"{len(queries)} queries loaded")
assert len(queries) == 10, f"expected 10 queries, got {len(queries)}"

10 queries loaded


In [4]:
for title, sql in queries:
    print("=" * 80)
    print(title)
    print("=" * 80)
    result = pd.read_sql(sql, conn)
    display(result)
    print()

1. Total revenue and profit by region, ranked highest revenue first.


,region,total_revenue,total_profit,num_orders
0,Sub-Saharan Africa,39672031.43,12183211.40,36
1,Europe,33368932.11,11082938.63,22
2,Asia,21347091.02,6113845.87,11
3,Australia and Oceania,14094265.13,4722160.03,11
4,Middle East and North Africa,14052706.58,5761191.86,10
5,Central America and the Caribbean,9170385.49,2846907.85,7
6,North America,5643356.55,1457942.76,3



2. Top 5 item types by total revenue.


,item_type,total_revenue,total_units_sold
0,Cosmetics,36601509.60,83718
1,Office Supplies,30585380.07,46967
2,Household,29889712.29,44727
3,Baby Food,10350327.60,40545
4,Clothes,7787292.80,71260



3. Profit margin (%) by item type -- which product lines are actually the most profitable


...

,region,country,item_type,sales_channel,order_priority,order_date,order_id,ship_date,units_sold,unit_price,unit_cost,total_revenue,total_cost,total_profit
0,Middle East and North Africa,Pakistan,Cosmetics,Offline,L,2013-07-05,231145322,2013-08-16,9892,437.2,263.33,4324782.4,2604860.36,1719922.04



9. Running (cumulative) total revenue over time, order by order -- a window function example.


,order_date,order_id,total_revenue,running_total_revenue
0,2010-02-02,385383069,247956.32,247956.32
1,2010-02-06,382392299,3162704.80,3410661.12
2,2010-05-07,686048400,54319.26,3464980.38
3,2010-05-28,669165933,2533654.00,5998634.38
4,2010-06-30,647876489,1082418.40,7081052.78
5,2010-10-24,166460740,5396577.27,12477630.05
6,2010-10-30,705784308,668356.48,13145986.53
7,2010-11-26,660643374,3458252.00,16604238.53
8,2010-12-23,617667090,22312.29,16626550.82
9,2010-12-30,441619336,2559474.10,19186024.92



10. Rank each region's item types by revenue within that region (window function: RANK).


,region,item_type,item_revenue,revenue_rank
0,Europe,Cosmetics,13159720.00,1
1,Sub-Saharan Africa,Office Supplies,10582813.71,1
2,Middle East and North Africa,Cosmetics,10324478.00,1
3,Asia,Household,8072701.60,1
4,Central America and the Caribbean,Household,5997054.98,1
5,North America,Household,4647149.58,1
6,Australia and Oceania,Cosmetics,4220728.80,1


## Notes

- Query 5 (processing time by priority) and query 9/10 (window functions) are the ones that
  most clearly need real SQL rather than a `pandas.groupby` -- `JULIANDAY` for date arithmetic,
  and `RANK() OVER (PARTITION BY ...)` for the per-region top item type.
- All ten queries above were executed against the actual SQLite database built from the CSV in
  this run, not hand-computed -- re-running this notebook top to bottom reproduces every number
  shown.